# Logistics Network Optimization: Balanced Swap Algorithm
**Author:** Aliaksandra Yakimovich  
**Stack:** Python (Pandas, NumPy, PySpark), SQL, Geospatial Analytics

## Business Problem
The current retail network suffers from inefficient store-to-hub assignments, leading to excessive mileage and high operational costs. 

**Objective:** Minimize total distribution mileage by reassigning stores to closer hubs while strictly maintaining the current workload (capacity) of each Regional Distribution Center (RDC).

In [0]:
import pandas as pd
import numpy as np
import random
from pyspark.sql import SparkSession

# Initialize Spark Session (Entry point for Databricks/Spark environments)
spark = SparkSession.builder.getOrCreate()

# Centralized Hub Configuration to avoid hardcoding in multiple places
hubs = {
    'Warszawa': (52.23, 21.01),
    'Lodz': (51.75, 19.45),
    'Bydgoszcz': (53.12, 18.00),
    'Poznan': (52.41, 16.92),
    'Wroclaw': (51.10, 17.03)
}

LAT_TO_KM = 111  # Degree to KM conversion for Poland (Latitude)
LON_TO_KM = 70   # Degree to KM conversion for Poland (Longitude)

## 1. Synthetic Data Generation: High-Density Logistics Network
In this stage, I am creating a synthetic dataset representing a complex retail supply chain. The network consists of 1,250 stores strategically clustered around 5 hubs to create "conflict zones" where optimization is non-obvious.

In [0]:
data = []
store_id_counter = 1000

for hub_name, coords in hubs.items():
    for _ in range(250):
        store_id_counter += 1
        is_border = random.random() < 0.20 
        
        if is_border:
            radius = random.uniform(0.6, 1.0) 
        else:
            radius = random.uniform(0.05, 0.4)
        
        angle = random.uniform(0, 2 * np.pi)
        lat = coords[0] + radius * np.cos(angle)
        lon = coords[1] + radius * np.sin(angle)
        
        p = random.random()
        if p < 0.85: trucks = 1
        elif p < 0.97: trucks = 2
        else: trucks = 3
        
        data.append([store_id_counter, lat, lon, hub_name, trucks])

df_final = pd.DataFrame(data, columns=['store_id', 'store_lat', 'store_lon', 'currently_assigned_hub', 'trucks_per_day'])
spark.createDataFrame(df_final).write.mode("overwrite").saveAsTable("workspace.default.retail_logistics_raw")

df_final = pd.DataFrame(data, columns=['store_id', 'store_lat', 'store_lon', 'currently_assigned_hub', 'trucks_per_day'])
spark.createDataFrame(df_final).write.mode("overwrite").saveAsTable("workspace.default.retail_logistics_raw")

print(f"Dataset generated: {len(df_final)} stores assigned to {len(HUB_COORDS)} hubs.")

Dataset generated: 1250 stores assigned to 5 hubs.


## 2. Geospatial Audit: Distance Matrix Calculation
Using SQL, I calculate the Euclidean distance from every store to all 5 possible Hubs. This "Distance Matrix" serves as the foundation for identifying logistics inefficiencies and "Value at Risk".

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.retail_distance_matrix AS
SELECT 
    s.*,
    ROUND(SQRT(POWER((s.store_lat - 52.23) * 111, 2) + POWER((s.store_lon - 21.01) * 70, 2)), 2) as dist_to_Warszawa,
    ROUND(SQRT(POWER((s.store_lat - 51.75) * 111, 2) + POWER((s.store_lon - 19.45) * 70, 2)), 2) as dist_to_Lodz,
    ROUND(SQRT(POWER((s.store_lat - 53.12) * 111, 2) + POWER((s.store_lon - 18.00) * 70, 2)), 2) as dist_to_Bydgoszcz,
    ROUND(SQRT(POWER((s.store_lat - 52.41) * 111, 2) + POWER((s.store_lon - 16.92) * 70, 2)), 2) as dist_to_Poznan,
    ROUND(SQRT(POWER((s.store_lat - 51.10) * 111, 2) + POWER((s.store_lon - 17.03) * 70, 2)), 2) as dist_to_Wroclaw
FROM workspace.default.retail_logistics_raw s;

num_affected_rows,num_inserted_rows


## 3. Optimization Engine: Balanced Pairwise Swap Algorithm
To optimize the network without disrupting Hub operations (avoiding overloading), I developed a **Balanced Swap Algorithm**. 

**Logic:** If a store is moved to a closer Hub, it must be swapped with another store moving in the opposite direction. This keeps the workload (capacity) for each Hub strictly neutral while reducing total mileage.

In [0]:
def fast_dist(lat1, lon1, lat2, lon2):
    return ((lat1 - lat2)**2 + (lon1 - lon2)**2)**0.5 * 111

def run_balanced_optimization(data):
    """
    Note: Using a pairwise swap heuristic for constrained optimization.
    Maintains capacity neutrality across the network.
    """
    optimized_data = data.copy()
    current_hubs = optimized_data['currently_assigned_hub'].tolist()
    lats = optimized_data['store_lat'].tolist()
    lons = optimized_data['store_lon'].tolist()
    trucks = optimized_data['trucks_per_day'].tolist()
    
    total_km_saved = 0
    swaps_count = 0

    for i in range(len(optimized_data)):
        for j in range(i + 1, len(optimized_data)):
            h_i, h_j = current_hubs[i], current_hubs[j]
            if h_i == h_j: continue

            c_i, c_j = hubs[h_i], hubs[h_j]

            # Current distance (Round trip)
            total_now = (fast_dist(lats[i], lons[i], c_i[0], c_i[1]) * trucks[i] * 2) + \
                        (fast_dist(lats[j], lons[j], c_j[0], c_j[1]) * trucks[j] * 2)

            # Distance after swap
            total_after = (fast_dist(lats[i], lons[i], c_j[0], c_j[1]) * trucks[i] * 2) + \
                          (fast_dist(lats[j], lons[j], c_i[0], c_i[1]) * trucks[j] * 2)

            if total_now > total_after:
                current_hubs[i], current_hubs[j] = h_j, h_i
                total_km_saved += (total_now - total_after)
                swaps_count += 1

    optimized_data['optimized_hub'] = current_hubs
    return optimized_data, total_km_saved, swaps_count

# Execution
df = spark.table("workspace.default.retail_logistics_raw").toPandas()
final_df, saved_km, count = run_balanced_optimization(df)

print(f"Total Swaps: {count}")
print(f"Daily Savings: {saved_km:.2f} KM")

Total Swaps: 32
Daily Savings: 1205.43 KM


## 4. Final Export & Visualization Prep
The optimized dataset is saved for Power BI integration. I also recalculate the final distance matrix to confirm the efficiency gains.

In [0]:
spark.createDataFrame(final_df).write.mode("overwrite").saveAsTable("workspace.default.retail_network_optimized")